In [1]:
import re
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef, recall_score
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from xgboost import XGBClassifier

from joblib import Parallel, delayed
from sklearn.base import clone
import os

In [2]:
# =========================================================
# 0. CONSTANTS: EPOCH LENGTH, SOP, SPH, MODELS, OUTPUT_DIR
# =========================================================
EPOCH_LEN = 10                # seconds per epoch
PREICTAL = 1800               # seconds
SOP = PREICTAL                     
N_SELECTED_FEATURES = 20      # SelectKBest (mutual information)
K_VALUES = (90,)                # alarm window: k consecutive epochs predicted preictal
THRESHOLD = 0.6
SAVE_DIR = f'D:\Results\Thres{THRESHOLD}'

MODELS = {
    'Random Forest': RandomForestClassifier(n_estimators=500, max_depth=15, min_samples_leaf=4,
                                            max_features='sqrt', random_state=42, n_jobs=1),
    'XGBoost': XGBClassifier(n_estimators=300, max_depth=10, learning_rate=0.05, subsample=0.8,
                             colsample_bytree=0.8, objective='binary:logistic',
                             eval_metric='logloss', random_state=42,n_jobs=1),
    'SVM': SVC(kernel='linear', C=100, random_state=42)
}
SEQ_KEYS = ['SeizureID', 'Class', 'Epoch']
BLOCK_KEYS = ['SeizureID', 'Epoch']
LABEL_MAP = {'preictal': 1, 'interictal': 0}
MERGE_MAP = {'chb01': 'chb21'}    # chb21 is merged into chb01 (same patient)

In [3]:
# =========================================================
# 1.1 SUPPORT FUNCTIONS: CALCULATIONS
# =========================================================
def labeling_for_SS(df_seizure):
    if df_seizure is None:
        return None
    if len(df_seizure)==0:
        return None
    df = df_seizure.copy()
    n = len(df)
    n_sop = min(SOP // EPOCH_LEN, n)
    cls = np.zeros(n, dtype=int)
    cls[n - n_sop :] = 1
    df['Class'] = cls
    return df

def check_nan(df_train, df_test, df_ss, threshold=0.05):
    """
    Handle NaN with a decision taken on the training set only.

    - train: share of rows with NaN <= threshold -> only drop NaN rows (in train, test and FAR sets);
             otherwise drop the feature columns with more than `threshold` NaN (in all three sets) first,
             then drop the remaining NaN rows.
    - FAR set: a time series, so removed epochs leave gaps. 'Discontinuity' = 1 for the epoch right after
      a removed epoch, 0 otherwise (used by calculate_far to reset the k-epoch window).

    Returns df_train, df_test, df_ss (with 'Discontinuity'), dropped_cols.
    """
    dropped_cols = []
    if df_train.isna().any(axis=1).mean() > threshold:
        nan_ratio = df_train.isna().mean()
        dropped_cols = nan_ratio[nan_ratio > threshold].index.tolist()
        df_train, df_test, df_ss = (df.drop(columns=dropped_cols) if df is not None else None for df in (df_train, df_test, df_ss))

    if df_ss is not None:
        removed = df_ss.isna().any(axis=1).to_numpy()
        after_removed = np.r_[False, removed[:-1]]
        df_ss = df_ss.assign(Discontinuity=after_removed.astype(int))[~removed]
        if df_ss.empty:
            df_ss = None
    return df_train.dropna(), df_test.dropna(), df_ss, dropped_cols

def calculate_far(y, k, refractory=SOP):
    if y is None:
        return None
    y = np.asarray(y)
    k_in_second = k*EPOCH_LEN
    if len(y) < (k_in_second + refractory) / EPOCH_LEN:
        return None
    k_epochs = int(k)
    refractory_len = int(refractory / EPOCH_LEN)
    false_alarms = refractory_epochs = idx = counter = 0
    while idx < len(y):
        counter = counter + 1 if y[idx] == 1 else 0
        if counter == k_epochs:
            false_alarms += 1
            refractory_epochs += len(y[idx + 1: idx + 1 + refractory_len])
            idx += refractory_len
            counter = 0
        idx += 1
    return false_alarms / ((len(y) - refractory_epochs) * EPOCH_LEN / 3600)

def calculate_true_alarm(y_pred, y_test, k, refractory=SOP):
    if y_test is None:
        return None
    k_in_second = k*EPOCH_LEN
    y_pred, y_test = np.asarray(y_pred), np.asarray(y_test)
    if len(y_test) < (k_in_second + refractory) / EPOCH_LEN:
        return None
    k_epochs = int(k)
    refractory_len = int(refractory / EPOCH_LEN)
    idx = counter = 0
    while idx < len(y_pred):
        counter = counter + 1 if y_pred[idx] == 1 else 0
        if counter == k_epochs:
            if y_test[idx] == 1:
                return True
            idx += refractory_len
            counter = 0
        idx += 1
    return False

def save_results(df_subject, file_path, model_name):
    df_subject.index.name = 'Subject'
    df_summary = pd.DataFrame({'Mean': df_subject.mean(axis=0), 'STD': df_subject.std(axis=0, ddof=0)})
    df_summary.index.name = 'Metrics'

    with open(file_path, 'a', encoding='utf-8') as f:
        f.write(f'Result for {model_name}\n')
        df_subject.to_csv(f)
        f.write(f'\nSummary across subjects {model_name}\n')
        df_summary.to_csv(f)


In [4]:
# =========================================================
# 1.2 LOAD DATA FINCTIONS
# =========================================================
def pivot_df(df, index, value_cols=None, columns='Band'):
    if value_cols is None:
        value_cols = [c for c in df.columns if c not in (*index, columns)]
    df_pivot = df.pivot(index=index, columns=columns, values=value_cols)
    df_pivot.columns = [f'{col}_{band}' for col, band in df_pivot.columns]
    return df_pivot.reset_index()

def merge_chb01_21(df_seq01, df_block01, df_seq21, df_block21):
    """
    chb01 and chb21 are the same patient (recorded 1.5 years apart): merge them into one subject.
    SeizureID of chb21 is shifted by the largest SeizureID of chb01, so every seizure keeps a unique ID
    and the LOSO folds treat the seizures of both recordings as seizures of one patient.
    The same offset is applied to the training frame (df_seq) and the testing frame (df_block),
    so the seizure <-> block correspondence is preserved.
    """
    offset = int(max(df_seq01['SeizureID'].max(), df_block01['SeizureID'].max()))
    df_seq21, df_block21 = df_seq21.copy(), df_block21.copy()
    df_seq21['SeizureID'] += offset
    df_block21['SeizureID'] += offset
    print(f'Merged chb21 into chb01: chb21 SeizureID shifted by {offset}')
    return (pd.concat([df_seq01, df_seq21], ignore_index=True),
            pd.concat([df_block01, df_block21], ignore_index=True))

def _find_col(df, name):
    """Tìm cột theo tên, không phân biệt hoa thường."""
    for c in df.columns:
        if c.strip().lower() == name:
            return c
    raise KeyError(f"Không thấy cột '{name}' trong {list(df.columns)}")

def _seizure_to_int(df, col='SeizureID'):
    if col in df.columns and not pd.api.types.is_numeric_dtype(df[col]):
        df[col] = (df[col].astype(str)
                          .str.extract(r'(\d+)(?!.*\d)', expand=False)
                          .astype(int))
    return df

def load_chb(folder, pivot=False):
    """
    Read csv files of a patient, return two dataframes which are training dataframe and the testing dataframe
    """
    folder = Path(folder)
    files = sorted(folder.glob("*.csv"))

    block_files = [f for f in files if re.search(r"_block_\d+", f.stem)]
    seq_files = [f for f in files if f not in block_files]

    # ---------- 1. training----------
    seq_files.sort(key=lambda f: (0 if "preictal" in f.stem else 1, f.stem))
    df_seq = pd.concat((pd.read_csv(f) for f in seq_files),
                       axis=0, ignore_index=True)
    # ---------- 2. testing----------
    def sort_key(f):
        blk = int(re.search(r"_block_(\d+)", f.stem).group(1))
        sz = int(re.search(r"seizure_(\d+)", f.stem).group(1))
        return (blk, sz)

    parts = []
    for f in sorted(block_files, key=sort_key):
        d = pd.read_csv(f)
        d = d.drop(columns=_find_col(d, "block"))
        parts.append(d)

    df_block = pd.concat(parts, axis=0, ignore_index=True)
    df_seq = _seizure_to_int(df_seq)
    df_block = _seizure_to_int(df_block)
    if pivot:
        df_seq = df_seq.drop(columns='Patient', errors='ignore')
        df_block = df_block.drop(columns='Patient', errors='ignore')
        df_seq = pivot_df(df_seq, SEQ_KEYS)
        df_block = pivot_df(df_block,BLOCK_KEYS)

    return df_seq, df_block

In [5]:
# =========================================================
# 2. LOSO
# =========================================================
def balance_train(df_train, seed):
    pre   = df_train[df_train['Class'] == 1]
    inter = df_train[df_train['Class'] == 0]
    if len(inter) > len(pre):
        inter = inter.sample(n=len(pre), random_state=int(seed))
    return pd.concat([pre, inter]).sort_index()

def evaluate_cv(df_seq, df_block, model, k, subject_id, scaler=None):
    # Each fold using data of a seizure for testing (calculating epoch-level metrics and FAR/h) and the remaining is used in training
    seizure_ids = np.sort(df_seq.loc[df_seq['Class'] == 1, 'SeizureID'].unique())  
    fold_metrics, fold_info = [], []    
    far_values, ss_values = [], []      

    for i, sid in enumerate(seizure_ids):   
        # Train/test is split directly from the df_seq
        is_test = df_seq['SeizureID'] == sid 
        train_data = df_seq[~is_test].drop(columns='SeizureID')
        test_data = df_seq[is_test].sort_values('Epoch', kind='stable').drop(columns='SeizureID')
        n_pre_train_raw, n_inter_train_raw = (train_data['Class'] == 1).sum(), (train_data['Class'] == 0).sum()
        n_pre_test_raw, n_inter_test_raw = (test_data['Class'] == 1).sum(), (test_data['Class'] == 0).sum()
    
        df_ss = labeling_for_SS(df_block[df_block['SeizureID'] == sid].drop(columns='SeizureID').sort_values('Epoch'))
        n_ss_before = len(df_ss) if df_ss is not None else None

        n_test_before = len(test_data)
        n_train_before = len(train_data)
        train_data, test_data, ss_data, dropped_cols = check_nan(train_data, test_data, df_ss)
        n_dropped_train = n_train_before - len(train_data)
        
        if (test_data['Class'] == 1).sum() == 0 or train_data['Class'].nunique() < 2:
            continue
        n_dropped_test = n_test_before-len(test_data)
        
        train_data = balance_train(train_data, seed=sid)
        n_pre_train, n_inter_train = int((train_data['Class'] == 1).sum()), int((train_data['Class'] == 0).sum())
        n_pre_test, n_inter_test = int((test_data['Class'] == 1).sum()), int((test_data['Class'] == 0).sum())
        fold_info.append({
            'Subject':subject_id,
            'Fold (SeizureID)': sid,
            'Total_Preictal_Train_Epochs': n_pre_train_raw,
            'Total_Preictal_Test_Epochs': n_pre_test_raw,
            'Total_Interictal_Train_Epochs': n_inter_train_raw,
            'Total_Interictal_Test_Epochs': n_inter_test_raw,
            'After_Dropped_Preictal_Train_Epochs': n_pre_train,
            'After_Dropped_Preictal_Test_Epochs': n_pre_test,
            'Balanced_Interictal_Train_Epochs': n_inter_train,
            'Balanced_Interictal_Test_Epochs': n_inter_test,
            'N_Dropped_Train_Epochs': n_dropped_train,
            'N_Dropped_Test_Epochs': n_dropped_test,
            'SS_test': len(ss_data) if ss_data is not None else None,
            'Preictal_Test_Train_%': n_pre_test / n_pre_train if n_pre_train else np.nan,
            'Interictal_Test_Train_%': n_inter_test / n_inter_train if n_inter_train else np.nan,
            'Dropped_Features': ', '.join(dropped_cols),
            'N_Dropped_Features': len(dropped_cols),
            'N_Dropped_SS_Epochs': n_ss_before - len(ss_data) if n_ss_before is not None and ss_data is not None else None})

        feature_cols = [c for c in train_data.columns if c not in ('Epoch', 'Class', 'Patient')]

        X_train, y_train = train_data[feature_cols], train_data['Class']
        X_test, y_test = test_data[feature_cols], test_data['Class'].to_numpy()
        if ss_data is not None:
            X_ss, y_ss = ss_data[feature_cols], ss_data['Class'].to_numpy()

        if scaler is not None:
            X_train = scaler.fit_transform(X_train)
            X_test = scaler.transform(X_test)
            if ss_data is not None:
                X_ss = scaler.transform(X_ss)

        selector = SelectKBest(score_func=lambda X, y: mutual_info_classif(X, y, random_state=42),
                               k=N_SELECTED_FEATURES)
        X_train = selector.fit_transform(X_train, y_train)

        model.fit(X_train, y_train)
        y_pred = model.predict(selector.transform(X_test))
        if ss_data is not None:
            y_pred_ss = model.predict(selector.transform(X_ss))

        sensitivity = recall_score(y_test, y_pred, pos_label=1)
        specificity = (recall_score(y_test, y_pred, pos_label=0)
                       if (y_test == 0).any() else np.nan)
        fold_metrics.append({
            'Accuracy': accuracy_score(y_test, y_pred),
            'Sensitivity': sensitivity,
            'Specificity': specificity,
            'F1': f1_score(y_test, y_pred, pos_label=1),
            'MCC': matthews_corrcoef(y_test, y_pred),
            'Balanced accuracy': (sensitivity + specificity) / 2})

        # FAR/h:
        far_per_hour = calculate_far(y_pred[y_test == 0], k)
        if far_per_hour is not None:
            far_values.append(far_per_hour)

        if ss_data is not None:
            # Seizure Sensitivity: 
            true_alarm = calculate_true_alarm(y_pred_ss, y_ss, k)
            if true_alarm is not None:
                ss_values.append(int(true_alarm))

    mean_metrics = pd.DataFrame(fold_metrics).mean(skipna=True).to_dict()
    mean_metrics['FAR/h'] = np.mean(far_values) if far_values else np.nan
    mean_metrics['Seizure Sensitivity'] = np.mean(ss_values) if ss_values else np.nan

    return mean_metrics, fold_info

In [6]:
# =========================================================
# 3. EXPERIMENTING FUNCTIONS
# =========================================================
def merge_on_keys(left, right, keys):
    keys = [c for c in keys if c in left.columns and c in right.columns]
    return left.merge(right, on=keys)

def _map_class(df):
    if 'Class' in df.columns and not pd.api.types.is_numeric_dtype(df['Class']):
        df['Class'] = df['Class'].astype(str).str.strip().str.lower().map(LABEL_MAP)
        if df['Class'].isna().any():
            raise ValueError(f"Nhãn Class lạ: {df['Class'].isna().sum()} dòng không map được")
    return df


def load_subject_root(root, subject_id):
    """Load one feature set of a subject; for chb01 also load chb21 and merge them."""
    df_seq, df_block = load_chb(os.path.join(root, subject_id), pivot=True)
    extra = MERGE_MAP.get(subject_id)
    if extra is not None:
        extra_dir = os.path.join(root, extra)
        if os.path.isdir(extra_dir):
            df_seq21, df_block21 = load_chb(extra_dir, pivot=True)
            df_seq, df_block = merge_chb01_21(df_seq, df_block, df_seq21, df_block21)
        else:
            print(f'WARNING: {extra_dir} not found, {subject_id} is NOT merged')

    df_seq, df_block = _map_class(df_seq), _map_class(df_block)   # preictal=1, interictal=0
    return df_seq, df_block


def load_subject_frames(subject_id, roots):
    """roots: Each root contains feature set of folders chbXX"""
    df_seq, df_block = load_subject_root(roots[0], subject_id)
    for root in roots[1:]:
        other_seq, other_block = load_subject_root(root, subject_id)
        df_seq = merge_on_keys(df_seq, other_seq, SEQ_KEYS)
        df_block = merge_on_keys(df_block, other_block, BLOCK_KEYS)
    return df_seq, df_block


def process_subject(subject_id, name, roots, k_values):
    try:
        print(f'PROCESSING {subject_id} [{name}]', flush=True)
        df_seq, df_block = load_subject_frames(subject_id, roots)
        results = {}
        for k in k_values:
            for model_name, model in MODELS.items():
                scaler = StandardScaler() if model_name == 'SVM' else None
                metrics, fold_info = evaluate_cv(
                    df_seq, df_block, clone(model), k, subject_id, scaler)
                results[(k, model_name)] = (
                    metrics, [{**row, 'Model': model_name} for row in fold_info])
        return subject_id, results
    except Exception as e:
        print(f'ERROR {subject_id}: {type(e).__name__}: {e}', flush=True)
        return subject_id, None


def run_experiment(name, roots, k_values=K_VALUES, n_jobs=-1):
    if isinstance(roots, str):
        roots = [roots]

    subjects = sorted(
        set.intersection(*(set(os.listdir(r)) for r in roots))-set(MERGE_MAP.values()))
    print(subjects)
    outputs = Parallel(n_jobs=n_jobs, verbose=10)(
        delayed(process_subject)(s, name, roots, k_values) for s in subjects)

    for k in k_values:
        out_dir = os.path.join(SAVE_DIR, f'k={k}')
        os.makedirs(out_dir, exist_ok=True)
        save_path = os.path.join(out_dir, f'{name}.csv')
        save_info_path = os.path.join(out_dir, f'Fold_infor_{name}.csv')
        if os.path.exists(save_path):
            os.remove(save_path)

        all_fold_info = []
        subject_ids = [sid for sid, _ in outputs]
        for model_name in MODELS:
            scores = [res[(k, model_name)][0] for _, res in outputs]
            for _, res in outputs:
                all_fold_info += res[(k, model_name)][1]
            if scores:
                save_results(pd.DataFrame(scores, index=subject_ids), save_path, model_name)

        if all_fold_info:
            pd.DataFrame(all_fold_info).to_csv(save_info_path, index=False)

In [7]:
# Run experiments for Handcrafted Features
base = r'D:\Downloads\Compressed\Handcrafted_Features'
run_experiment(name='Handcrafted',  roots=base,n_jobs=-1)

['chb01', 'chb02', 'chb03', 'chb04', 'chb05', 'chb06', 'chb07', 'chb09', 'chb10', 'chb13', 'chb14', 'chb18', 'chb20', 'chb22', 'chb23']


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 out of  15 | elapsed:  1.2min remaining:  7.6min
[Parallel(n_jobs=-1)]: Done   4 out of  15 | elapsed:  1.7min remaining:  4.8min
[Parallel(n_jobs=-1)]: Done   6 out of  15 | elapsed:  2.0min remaining:  3.0min
[Parallel(n_jobs=-1)]: Done   8 out of  15 | elapsed:  2.1min remaining:  1.9min
[Parallel(n_jobs=-1)]: Done  10 out of  15 | elapsed:  2.3min remaining:  1.1min
[Parallel(n_jobs=-1)]: Done  12 out of  15 | elapsed:  2.7min remaining:   40.3s
[Parallel(n_jobs=-1)]: Done  15 out of  15 | elapsed:  6.5min finished


In [8]:
# Run experiments for Graph-based Features
base = r'D:\Downloads\Compressed\Graph-based_Feature'
run_experiment(name='PLI',  roots=os.path.join(base, f'PLI_features_{THRESHOLD}'), n_jobs=-1)
run_experiment(name='wPLI', roots=os.path.join(base, f'wPLI_features_{THRESHOLD}'),n_jobs=-1)

['chb01', 'chb02', 'chb03', 'chb04', 'chb05', 'chb06', 'chb07', 'chb09', 'chb10', 'chb13', 'chb14', 'chb18', 'chb20', 'chb22', 'chb23']


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 out of  15 | elapsed:   30.7s remaining:  3.3min
[Parallel(n_jobs=-1)]: Done   4 out of  15 | elapsed:   53.6s remaining:  2.5min
[Parallel(n_jobs=-1)]: Done   6 out of  15 | elapsed:  1.0min remaining:  1.6min
[Parallel(n_jobs=-1)]: Done   8 out of  15 | elapsed:  1.2min remaining:  1.0min
[Parallel(n_jobs=-1)]: Done  10 out of  15 | elapsed:  1.3min remaining:   37.6s
[Parallel(n_jobs=-1)]: Done  12 out of  15 | elapsed:  1.5min remaining:   22.2s
[Parallel(n_jobs=-1)]: Done  15 out of  15 | elapsed:  4.7min finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 12 concurrent workers.


['chb01', 'chb02', 'chb03', 'chb04', 'chb05', 'chb06', 'chb07', 'chb09', 'chb10', 'chb13', 'chb14', 'chb18', 'chb20', 'chb22', 'chb23']


[Parallel(n_jobs=-1)]: Done   2 out of  15 | elapsed:   28.6s remaining:  3.1min
[Parallel(n_jobs=-1)]: Done   4 out of  15 | elapsed:   51.3s remaining:  2.4min
[Parallel(n_jobs=-1)]: Done   6 out of  15 | elapsed:   57.9s remaining:  1.4min
[Parallel(n_jobs=-1)]: Done   8 out of  15 | elapsed:  1.1min remaining:   59.1s
[Parallel(n_jobs=-1)]: Done  10 out of  15 | elapsed:  1.2min remaining:   35.5s
[Parallel(n_jobs=-1)]: Done  12 out of  15 | elapsed:  1.4min remaining:   21.4s
[Parallel(n_jobs=-1)]: Done  15 out of  15 | elapsed:  4.6min finished


In [9]:
# Run experiments for hybrid Features
base_graph = r'D:\Downloads\Compressed\Graph-based_Feature'
base_handcraft = r'D:\Downloads\Compressed\Handcrafted_Features'
run_experiment(name='Hybrid-PLI',  roots=[os.path.join(base_graph, f'PLI_features_{THRESHOLD}'), base_handcraft], n_jobs=-1)
run_experiment(name='Hybrid-wPLI', roots=[os.path.join(base, f'wPLI_features_{THRESHOLD}'), base_handcraft],n_jobs=-1)

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 12 concurrent workers.


['chb01', 'chb02', 'chb03', 'chb04', 'chb05', 'chb06', 'chb07', 'chb09', 'chb10', 'chb13', 'chb14', 'chb18', 'chb20', 'chb22', 'chb23']


[Parallel(n_jobs=-1)]: Done   2 out of  15 | elapsed:  1.7min remaining: 11.0min
[Parallel(n_jobs=-1)]: Done   4 out of  15 | elapsed:  2.5min remaining:  6.8min
[Parallel(n_jobs=-1)]: Done   6 out of  15 | elapsed:  2.8min remaining:  4.3min
[Parallel(n_jobs=-1)]: Done   8 out of  15 | elapsed:  3.0min remaining:  2.6min
[Parallel(n_jobs=-1)]: Done  10 out of  15 | elapsed:  3.2min remaining:  1.6min
[Parallel(n_jobs=-1)]: Done  12 out of  15 | elapsed:  3.8min remaining:   57.3s
[Parallel(n_jobs=-1)]: Done  15 out of  15 | elapsed:  8.4min finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 12 concurrent workers.


['chb01', 'chb02', 'chb03', 'chb04', 'chb05', 'chb06', 'chb07', 'chb09', 'chb10', 'chb13', 'chb14', 'chb18', 'chb20', 'chb22', 'chb23']


[Parallel(n_jobs=-1)]: Done   2 out of  15 | elapsed:  1.7min remaining: 11.2min
[Parallel(n_jobs=-1)]: Done   4 out of  15 | elapsed:  2.5min remaining:  6.8min
[Parallel(n_jobs=-1)]: Done   6 out of  15 | elapsed:  2.7min remaining:  4.1min
[Parallel(n_jobs=-1)]: Done   8 out of  15 | elapsed:  2.9min remaining:  2.5min
[Parallel(n_jobs=-1)]: Done  10 out of  15 | elapsed:  3.2min remaining:  1.6min
[Parallel(n_jobs=-1)]: Done  12 out of  15 | elapsed:  3.8min remaining:   56.8s
[Parallel(n_jobs=-1)]: Done  15 out of  15 | elapsed:  8.5min finished
